In [2]:
import os
import pandas as pd

from socceraction.data.statsbomb import StatsBombLoader
import socceraction.spadl.statsbomb as sb_spadl
import socceraction.spadl as spadl
import socceraction.vaep.features as vaep_features
import socceraction.vaep.labels as vaep_labels

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

## 1) Configurazione
Imposta il percorso della cartella dati e seleziona competizione e stagione.

Esempio: `competition_id=43`, `season_id=3` = World Cup 2018.

In [3]:
# Cartella con i JSON StatsBomb (nel tuo repo: data/)
data_root = 'data'

# Inizializza loader locale
SBL = StatsBombLoader(getter='local', root=data_root)

df_competitions = SBL.competitions()
print('competitions:', df_competitions.shape)
df_competitions.head()

competitions: (75, 6)


,season_id,competition_id,competition_name,country_name,competition_gender,season_name
0,281,9,1. Bundesliga,Germany,male,2023/2024
1,27,9,1. Bundesliga,Germany,male,2015/2016
2,107,1267,African Cup of Nations,Africa,male,2023
3,4,16,Champions League,Europe,male,2018/2019
4,1,16,Champions League,Europe,male,2017/2018


## 2) Caricare le partite (games) ed eventi (events)
Con `socceraction>=1.x` si usa `SBL.games(...)` e poi `SBL.events(game_id)`.

In [4]:
all_games = []
for i, comp in df_competitions.iterrows():
    cid = comp.competition_id
    sid = comp.season_id
    try:
        games = SBL.games(competition_id=cid, season_id=sid)
        all_games.append(games)
        print(f'loaded {len(games)} games for {comp.competition_name} {comp.season_name}')
    except Exception as e:
        print(f'Could not load games for {comp.competition_name} {comp.season_name}: {e}')

df_games = pd.concat(all_games, ignore_index=True) if all_games else pd.DataFrame()
print('\ntotal games loaded:', df_games.shape)
df_games.head()

loaded 34 games for 1. Bundesliga 2023/2024
loaded 306 games for 1. Bundesliga 2015/2016
loaded 52 games for African Cup of Nations 2023
loaded 1 games for Champions League 2018/2019
loaded 1 games for Champions League 2017/2018
loaded 1 games for Champions League 2016/2017
loaded 1 games for Champions League 2015/2016
loaded 1 games for Champions League 2014/2015
loaded 1 games for Champions League 2013/2014
loaded 1 games for Champions League 2012/2013
loaded 1 games for Champions League 2011/2012
loaded 1 games for Champions League 2010/2011
loaded 1 games for Champions League 2009/2010
loaded 1 games for Champions League 2008/2009
loaded 1 games for Champions League 2006/2007
loaded 1 games for Champions League 2004/2005
loaded 1 games for Champions League 2003/2004
loaded 1 games for Champions League 1999/2000
loaded 1 games for Champions League 1972/1973
loaded 1 games for Champions League 1971/1972
loaded 1 games for Champions League 1970/1971
loaded 32 games for Copa America 20

,game_id,season_id,competition_id,competition_stage,game_day,game_date,home_team_id,away_team_id,home_score,away_score,venue,referee
0,3895302,281,9,Regular Season,29,2024-04-14 17:30:00,904,176,5,0,BayArena,Harm Osmers
1,3895292,281,9,Regular Season,28,2024-04-06 15:30:00,190,904,0,1,Stadion An der Alten Försterei,Benjamin Brand
2,3895333,281,9,Regular Season,32,2024-05-05 18:30:00,184,904,1,5,Deutsche Bank Park,Christian Dingert
3,3895340,281,9,Regular Season,33,2024-05-12 20:30:00,868,904,0,5,Vonovia Ruhrstadion,Benjamin Brand
4,3895348,281,9,Regular Season,34,2024-05-18 16:30:00,904,172,2,1,BayArena,Matthias Jöllenbeck


## 3) Convertire in azioni SPADL
`convert_to_actions(events, home_team_id)` converte gli eventi StatsBomb in azioni SPADL per una singola partita.

Nota: per evitare warning di inferenza, impostiamo `xy_fidelity_version=1` e `shot_fidelity_version=1` (valori standard per gli open-data).

In [5]:
all_actions = []
failed_games = []

for i, game in df_games.reset_index(drop=True).iterrows():
    game_id = int(game.game_id)
    home_team_id = int(game.home_team_id)
    try:
        events = SBL.events(game_id)
        actions = sb_spadl.convert_to_actions(
            events,
            home_team_id=home_team_id,
            xy_fidelity_version=1,
            shot_fidelity_version=1,
        )
        all_actions.append(actions)
    except Exception as e:
        failed_games.append((game_id, str(e)))

    if (i + 1) % 100 == 0:
        print(f'converted {i+1}/{len(df_games)} games')

df_actions = pd.concat(all_actions, ignore_index=True) if len(all_actions) else pd.DataFrame()
print('actions:', df_actions.shape)

print('failed games:', len(failed_games))
df_actions.head()

converted 100/3464 games
converted 200/3464 games
converted 300/3464 games
converted 400/3464 games
converted 500/3464 games
converted 600/3464 games
converted 700/3464 games
converted 800/3464 games
converted 900/3464 games
converted 1000/3464 games
converted 1100/3464 games
converted 1200/3464 games
converted 1300/3464 games
converted 1400/3464 games
converted 1500/3464 games
converted 1600/3464 games
converted 1700/3464 games
converted 1800/3464 games
converted 1900/3464 games
converted 2000/3464 games
converted 2100/3464 games
converted 2200/3464 games
converted 2300/3464 games
converted 2400/3464 games
converted 2500/3464 games
converted 2600/3464 games
converted 2700/3464 games
converted 2800/3464 games
converted 2900/3464 games
converted 3000/3464 games
converted 3100/3464 games
converted 3200/3464 games
converted 3300/3464 games
converted 3400/3464 games
actions: (7005279, 14)
failed games: 0


,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id
0,3895302,221b0c8d-6386-4ae8-bb4a-a1dc98742312,1,3.417,176,34870.0,52.0625,33.660,53.8125,34.340,0,1,5,0
1,3895302,77809242-1460-4395-8779-94a0cfc275b1,1,3.870,176,12299.0,53.8125,34.340,53.8125,34.085,21,1,0,1
2,3895302,ff56e821-21e9-4cef-ba2a-7eb5eb3769c6,1,4.732,176,12299.0,53.8125,34.085,74.7250,36.295,0,1,5,2
3,3895302,cf9088bc-7e59-4d57-8ac5-31658da858bb,1,6.728,176,31100.0,74.7250,36.295,74.7250,36.720,21,1,0,3
4,3895302,4464cb75-f45f-4508-8444-2560d1625d06,1,7.622,176,31100.0,74.7250,36.720,76.3875,53.805,0,1,5,4


## 4) Caricare teams e players
Le tabelle `teams` e `players` vengono aggregate su tutte le partite selezionate.

In [6]:
all_teams = []
all_players = []

for i, game_id in enumerate(df_games.game_id.astype(int).tolist()):
    all_teams.append(SBL.teams(game_id))
    all_players.append(SBL.players(game_id))

    if (i + 1) % 25 == 0:
        
        print(f'loaded teams/players for {i+1}/{len(df_games)} games')

df_teams = pd.concat(all_teams, ignore_index=True).drop_duplicates(subset=['team_id']).reset_index(drop=True)
df_players = pd.concat(all_players, ignore_index=True).drop_duplicates(subset=['player_id']).reset_index(drop=True)

print('teams:', df_teams.shape)
print('players:', df_players.shape)
df_teams.head()

loaded teams/players for 25/3464 games
loaded teams/players for 50/3464 games
loaded teams/players for 75/3464 games
loaded teams/players for 100/3464 games
loaded teams/players for 125/3464 games
loaded teams/players for 150/3464 games
loaded teams/players for 175/3464 games
loaded teams/players for 200/3464 games
loaded teams/players for 225/3464 games
loaded teams/players for 250/3464 games
loaded teams/players for 275/3464 games
loaded teams/players for 300/3464 games
loaded teams/players for 325/3464 games
loaded teams/players for 350/3464 games
loaded teams/players for 375/3464 games
loaded teams/players for 400/3464 games
loaded teams/players for 425/3464 games
loaded teams/players for 450/3464 games
loaded teams/players for 475/3464 games
loaded teams/players for 500/3464 games
loaded teams/players for 525/3464 games
loaded teams/players for 550/3464 games
loaded teams/players for 575/3464 games
loaded teams/players for 600/3464 games
loaded teams/players for 625/3464 games
loa

,team_id,team_name
0,904,Bayer Leverkusen
1,176,Werder Bremen
2,190,Union Berlin
3,184,Eintracht Frankfurt
4,868,Bochum


## 5) Salvare `spadl.h5`
Creiamo (o sovrascriviamo) il file `spadl.h5` con le chiavi usate tipicamente in `socceraction`: `actions`, `games`, `teams`, `players`.

In [7]:
import tables

# Ensure no stale open handles to the HDF5 file (common when HDFStore was opened without closing)
tables.file._open_files.close_all()

out_file = 'spadl.h5'
if os.path.exists(out_file):
    os.remove(out_file)

# Avoid PyTables pickling warnings: enforce consistent string values in text columns
_text_cols_players = ['player_name', 'nickname', 'starting_position_name']
for _col in _text_cols_players:
    if _col in df_players.columns:
        # Convert mixed/object values (including NaN) to plain strings
        df_players[_col] = df_players[_col].fillna('').astype(str)

_text_cols_games = ['competition_stage', 'venue', 'referee']
for _col in _text_cols_games:
    if _col in df_games.columns:
        df_games[_col] = df_games[_col].fillna('').astype(str)

# Write everything via a single HDFStore context to guarantee clean close()
with pd.HDFStore(out_file, mode='w') as store:
    store.put('actions', df_actions, format='table')
    store.put('games', df_games, format='fixed')
    store.put('teams', df_teams, format='fixed')
    store.put('players', df_players, format='table')

    print('Saved', out_file)
    print('keys:', store.keys())

Saved spadl.h5
keys: ['/actions', '/games', '/players', '/teams']


In [8]:
# (opzionale) mostra eventuali partite fallite
if failed_games:
    pd.DataFrame(failed_games, columns=['game_id', 'error']).head(10)
else:
    print('No failed games')

No failed games


## 6) Generare `features.h5` (VAEP features)
Calcoliamo le feature di stato usando `socceraction.vaep.features.gamestates(...)` e le trasformazioni standard (one-hot + geometriche).
Salviamo tutto in `features.h5` con colonne `game_id` e `action_id` per allineare facilmente con le label.

In [9]:
# Genera features.h5 + labels.h5 (VAEP) a partire da spadl.h5

import pandas as pd

import socceraction.spadl as spadl
import socceraction.spadl.config as spadlcfg
import socceraction.vaep.features as vaep_features
import socceraction.vaep.labels as vaep_labels

spadl_file = 'spadl.h5'
features_file = 'features.h5'
labels_file = 'labels.h5'

import os as _os
if not _os.path.exists(spadl_file):
    raise FileNotFoundError(f"{spadl_file} non trovato. Esegui prima la cella che salva spadl.h5.")

def _as_series(obj, name: str) -> pd.Series:
    """Compatibility shim: socceraction labels may return Series or 1-col DataFrame."""
    if isinstance(obj, pd.DataFrame):
        if name in obj.columns and obj.shape[1] == 1:
            s = obj[name]
        elif obj.shape[1] == 1:
            s = obj.iloc[:, 0]
        else:
            # Fallback: take first column
            s = obj.iloc[:, 0]
    else:
        s = obj
    return s.rename(name)

# Carica actions e games e normalizza la direzione di gioco (left-to-right per la squadra di casa)
with pd.HDFStore(spadl_file, mode='r') as store:
    actions = store['actions'].copy()
    games = store['games'].copy()

# Assicura action_id per ogni partita (alcune pipeline non lo persistono)
if 'action_id' not in actions.columns:
    actions = actions.sort_values(['game_id', 'period_id', 'time_seconds']).copy()
    actions['action_id'] = actions.groupby('game_id').cumcount().astype(int)

# Alcune versioni di socceraction.vaep.features richiedono type_name (oltre a type_id)
if 'type_name' not in actions.columns and 'type_id' in actions.columns:
    _actiontype_map = dict(enumerate(spadlcfg.actiontypes))
    actions['type_name'] = actions['type_id'].map(_actiontype_map).astype(str)

# Allinea direzione e ordina
actions = actions.merge(games[['game_id', 'home_team_id']], on='game_id', how='left')
actions = actions.sort_values(['game_id', 'period_id', 'time_seconds', 'action_id']).reset_index(drop=True)
actions = actions.groupby('game_id', group_keys=False).apply(
    lambda x: spadl.play_left_to_right(x.drop(columns=['home_team_id']), home_team_id=int(x.home_team_id.iloc[0]))
).reset_index(drop=True)

nb_prev_actions = 3

# Feature functions standard VAEP
xfns = [
    vaep_features.actiontype_onehot,
    vaep_features.result_onehot,
    vaep_features.bodypart_onehot,
    vaep_features.startlocation,
    vaep_features.endlocation,
    vaep_features.movement,
    vaep_features.space_delta,
    vaep_features.goalscore,
    vaep_features.time,
 ]

features_parts = []
labels_parts = []

for game_id in games.game_id.astype(int).tolist():
    game_actions = actions[actions.game_id == game_id].reset_index(drop=True)
    if len(game_actions) == 0:
        continue

    gs = vaep_features.gamestates(game_actions, nb_prev_actions)
    X_game = pd.concat([fn(gs) for fn in xfns], axis=1)
    X_game.insert(0, 'game_id', game_id)
    X_game.insert(1, 'action_id', game_actions.action_id.astype(int).values)
    features_parts.append(X_game)

    y_scores = _as_series(vaep_labels.scores(game_actions), 'scores')
    y_concedes = _as_series(vaep_labels.concedes(game_actions), 'concedes')
    Y_game = pd.concat([y_scores, y_concedes], axis=1)
    Y_game.insert(0, 'game_id', game_id)
    Y_game.insert(1, 'action_id', game_actions.action_id.astype(int).values)
    labels_parts.append(Y_game)

df_features = pd.concat(features_parts, ignore_index=True) if features_parts else pd.DataFrame()
df_labels = pd.concat(labels_parts, ignore_index=True) if labels_parts else pd.DataFrame()

# Scrivi su disco (sovrascrive)
for f in [features_file, labels_file]:
    if _os.path.exists(f):
        _os.remove(f)

with pd.HDFStore(features_file, mode='w') as store:
    store.put('features', df_features, format='table')
    print('Saved', features_file, '->', df_features.shape)

with pd.HDFStore(labels_file, mode='w') as store:
    store.put('labels', df_labels, format='table')
    print('Saved', labels_file, '->', df_labels.shape)

df_features.head()

Saved features.h5 -> (7005279, 140)
Saved labels.h5 -> (7005279, 4)


,game_id,action_id,actiontype_pass_a0,actiontype_cross_a0,actiontype_throw_in_a0,actiontype_freekick_crossed_a0,actiontype_freekick_short_a0,actiontype_corner_crossed_a0,actiontype_corner_short_a0,actiontype_take_on_a0,...,goalscore_diff,period_id_a0,time_seconds_a0,time_seconds_overall_a0,period_id_a1,time_seconds_a1,time_seconds_overall_a1,period_id_a2,time_seconds_a2,time_seconds_overall_a2
0,3895302,0,True,False,False,False,False,False,False,False,...,0,1,3.417,3.417,1.0,3.417,3.417,1.0,3.417,3.417
1,3895302,1,False,False,False,False,False,False,False,False,...,0,1,3.870,3.870,1.0,3.417,3.417,1.0,3.417,3.417
2,3895302,2,True,False,False,False,False,False,False,False,...,0,1,4.732,4.732,1.0,3.870,3.870,1.0,3.417,3.417
3,3895302,3,False,False,False,False,False,False,False,False,...,0,1,6.728,6.728,1.0,4.732,4.732,1.0,3.870,3.870
4,3895302,4,True,False,False,False,False,False,False,False,...,0,1,7.622,7.622,1.0,6.728,6.728,1.0,4.732,4.732


## Prossimi passi
Ora che `spadl.h5` esiste, puoi generare `features.h5` e `labels.h5` (scores/concedes) e poi addestrare i modelli supervisionati (`p_score`, `p_concede`).

In [10]:
# # Addestra modelli baseline p_score e p_concede (Logistic Regression)

# import pandas as pd

# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import roc_auc_score, brier_score_loss

# features_file = 'features.h5'
# labels_file = 'labels.h5'

# with pd.HDFStore(features_file, mode='r') as store:
#     df_features = store['features'].copy()

# with pd.HDFStore(labels_file, mode='r') as store:
#     df_labels = store['labels'].copy()

# # Join su (game_id, action_id)
# df = df_features.merge(df_labels, on=['game_id', 'action_id'], how='inner')
# print('joined:', df.shape)

# # Split semplice by game_id (no leakage temporale intra-partita)
# game_ids = df['game_id'].drop_duplicates().sort_values().tolist()
# cut = int(0.8 * len(game_ids))
# train_games = set(game_ids[:cut])
# test_games = set(game_ids[cut:])

# train = df[df.game_id.isin(train_games)].copy()
# test = df[df.game_id.isin(test_games)].copy()
# print('train:', train.shape, 'test:', test.shape)

# # Feature matrix: tutte le colonne tranne id e labels
# drop_cols = {'game_id', 'action_id', 'scores', 'concedes'}
# feature_cols = [c for c in df_features.columns if c not in drop_cols]

# X_train = train[feature_cols].fillna(0)
# X_test = test[feature_cols].fillna(0)

# y_score_train = train['scores'].astype(int)
# y_score_test = test['scores'].astype(int)
# y_concede_train = train['concedes'].astype(int)
# y_concede_test = test['concedes'].astype(int)

# # Modelli baseline (puoi sostituire con XGBoost/CatBoost se preferisci)
# p_score = LogisticRegression(max_iter=100, n_jobs=None)
# p_concede = LogisticRegression(max_iter=100, n_jobs=None)

# p_score.fit(X_train, y_score_train)
# p_concede.fit(X_train, y_concede_train)

# p_score_hat = p_score.predict_proba(X_test)[:, 1]
# p_concede_hat = p_concede.predict_proba(X_test)[:, 1]

# print('p_score  AUC:', roc_auc_score(y_score_test, p_score_hat), 'Brier:', brier_score_loss(y_score_test, p_score_hat))
# print('p_concede AUC:', roc_auc_score(y_concede_test, p_concede_hat), 'Brier:', brier_score_loss(y_concede_test, p_concede_hat))

## 7) Creare un unico dataset
Uniamo le tabelle `actions`, `games`, `teams` e `competitions` per avere un dataset completo con tutte le informazioni.

In [11]:
# Unisci actions e games
df_full = pd.merge(df_actions, df_games, on='game_id', how='left')

# Unisci con le informazioni sulle squadre
df_full = pd.merge(df_full, df_teams, on='team_id', how='left')

# Unisci con le informazioni sulla competizione
df_full = pd.merge(df_full, df_competitions, on=['competition_id', 'season_id'], how='left')

print('full dataset shape:', df_full.shape)
sum(df_full.team_name=='Italy')

full dataset shape: (7005279, 30)


14774

In [12]:
print(df_full.columns)

Index(['game_id', 'original_event_id', 'period_id', 'time_seconds', 'team_id',
       'player_id', 'start_x', 'start_y', 'end_x', 'end_y', 'type_id',
       'result_id', 'bodypart_id', 'action_id', 'season_id', 'competition_id',
       'competition_stage', 'game_day', 'game_date', 'home_team_id',
       'away_team_id', 'home_score', 'away_score', 'venue', 'referee',
       'team_name', 'competition_name', 'country_name', 'competition_gender',
       'season_name'],
      dtype='object')


In [13]:
# Create a table with the name of the league and the year
league_year_table = df_full[['competition_name', 'season_name']].drop_duplicates().reset_index(drop=True)
print(league_year_table)

          competition_name season_name
0            1. Bundesliga   2023/2024
1            1. Bundesliga   2015/2016
2   African Cup of Nations        2023
3         Champions League   2018/2019
4         Champions League   2017/2018
..                     ...         ...
70      UEFA Europa League   1988/1989
71       UEFA Women's Euro        2025
72       UEFA Women's Euro        2022
73       Women's World Cup        2023
74       Women's World Cup        2019

[75 rows x 2 columns]


In [14]:
# Set display options to show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

# Display the full table
print(league_year_table)

           competition_name season_name
0             1. Bundesliga   2023/2024
1             1. Bundesliga   2015/2016
2    African Cup of Nations        2023
3          Champions League   2018/2019
4          Champions League   2017/2018
5          Champions League   2016/2017
6          Champions League   2015/2016
7          Champions League   2014/2015
8          Champions League   2013/2014
9          Champions League   2012/2013
10         Champions League   2011/2012
11         Champions League   2010/2011
12         Champions League   2009/2010
13         Champions League   2008/2009
14         Champions League   2006/2007
15         Champions League   2004/2005
16         Champions League   2003/2004
17         Champions League   1999/2000
18         Champions League   1972/1973
19         Champions League   1971/1972
20         Champions League   1970/1971
21             Copa America        2024
22             Copa del Rey   1983/1984
23             Copa del Rey   1982/1983


## 8) Creare il dataset completo
Uniamo tutte le informazioni in un unico DataFrame per l'analisi. Questo include:
- Azioni SPADL
- Dati della partita (competizione, stagione, etc.)
- Informazioni su squadre e giocatori
- Etichette VAEP (scores, concedes)

In [15]:
# Carica i dati necessari
with pd.HDFStore('spadl.h5', 'r') as store:
    df_actions = store.get('actions')
    df_games = store.get('games')
    df_teams = store.get('teams')
    df_players = store.get('players')

with pd.HDFStore('labels.h5', 'r') as store:
    df_labels = store.get('labels')

df_competitions = SBL.competitions()

def _ensure_cols_from_index(df: pd.DataFrame, required_cols: list[str]) -> pd.DataFrame:
    """If required cols are in the index (common with HDF), reset_index to expose them."""
    if not set(required_cols).issubset(df.columns):
        df = df.reset_index()
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns {missing}. Available: {list(df.columns)}")
    return df

# Assicura chiavi di join come colonne (non come index)
df_actions = _ensure_cols_from_index(df_actions, ['game_id', 'action_id'])
df_games = _ensure_cols_from_index(df_games, ['game_id'])
df_teams = _ensure_cols_from_index(df_teams, ['team_id'])
df_players = _ensure_cols_from_index(df_players, ['player_id'])
df_labels = _ensure_cols_from_index(df_labels, ['game_id', 'action_id'])
df_competitions = _ensure_cols_from_index(df_competitions, ['competition_id', 'season_id'])

def _drop_overlaps(right: pd.DataFrame, left_cols: pd.Index, join_keys: set[str]) -> pd.DataFrame:
    """Drop non-key columns in right that would collide with left during merge."""
    overlaps = (set(right.columns) & set(left_cols)) - set(join_keys)
    if overlaps:
        right = right.drop(columns=sorted(overlaps))
    return right

# 1) Merge actions con games
merged_df = pd.merge(df_actions, df_games, on='game_id', how='left')

# 2) Merge con teams (evita collisioni di colonne)
df_teams_m = _drop_overlaps(df_teams, merged_df.columns, join_keys={'team_id'})
merged_df = pd.merge(merged_df, df_teams_m, on='team_id', how='left')

# 3) Merge con players (evita che game_id diventi game_id_x/game_id_y)
df_players_m = _drop_overlaps(df_players, merged_df.columns, join_keys={'player_id'})
merged_df = pd.merge(merged_df, df_players_m, on='player_id', how='left')

# 4) Merge con competitions
df_comp_m = _drop_overlaps(df_competitions, merged_df.columns, join_keys={'competition_id', 'season_id'})
merged_df = pd.merge(merged_df, df_comp_m, on=['competition_id', 'season_id'], how='left')

# 5) Merge con labels (tieni solo keys + colonne label)
label_cols = [c for c in df_labels.columns if c in {'game_id', 'action_id', 'scores', 'concedes'}]
df_labels_m = df_labels[label_cols].copy()
df_labels_m = _drop_overlaps(df_labels_m, merged_df.columns, join_keys={'game_id', 'action_id'})
merged_df = pd.merge(merged_df, df_labels_m, on=['game_id', 'action_id'], how='left')

print("Shape del dataset completo:", merged_df.shape)
print("Colonne disponibili:", merged_df.columns.tolist())
merged_df.head()

Shape del dataset completo: (7005279, 39)
Colonne disponibili: ['game_id', 'original_event_id', 'period_id', 'time_seconds', 'team_id', 'player_id', 'start_x', 'start_y', 'end_x', 'end_y', 'type_id', 'result_id', 'bodypart_id', 'action_id', 'season_id', 'competition_id', 'competition_stage', 'game_day', 'game_date', 'home_team_id', 'away_team_id', 'home_score', 'away_score', 'venue', 'referee', 'team_name', 'player_name', 'nickname', 'jersey_number', 'is_starter', 'starting_position_id', 'starting_position_name', 'minutes_played', 'competition_name', 'country_name', 'competition_gender', 'season_name', 'scores', 'concedes']


,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id,season_id,competition_id,competition_stage,game_day,game_date,home_team_id,away_team_id,home_score,away_score,venue,referee,team_name,player_name,nickname,jersey_number,is_starter,starting_position_id,starting_position_name,minutes_played,competition_name,country_name,competition_gender,season_name,scores,concedes
0,3895302,221b0c8d-6386-4ae8-bb4a-a1dc98742312,1,3.417,176,34870.0,52.0625,33.660,53.8125,34.340,0,1,5,0,281,9,Regular Season,29,2024-04-14 17:30:00,904,176,5,0,BayArena,Harm Osmers,Werder Bremen,Nick Woltemade,,29,True,22,Right Center Forward,72,1. Bundesliga,Germany,male,2023/2024,False,False
1,3895302,77809242-1460-4395-8779-94a0cfc275b1,1,3.870,176,12299.0,53.8125,34.340,53.8125,34.085,21,1,0,1,281,9,Regular Season,29,2024-04-14 17:30:00,904,176,5,0,BayArena,Harm Osmers,Werder Bremen,Marvin Ducksch,,7,True,24,Left Center Forward,93,1. Bundesliga,Germany,male,2023/2024,False,False
2,3895302,ff56e821-21e9-4cef-ba2a-7eb5eb3769c6,1,4.732,176,12299.0,53.8125,34.085,74.7250,36.295,0,1,5,2,281,9,Regular Season,29,2024-04-14 17:30:00,904,176,5,0,BayArena,Harm Osmers,Werder Bremen,Marvin Ducksch,,7,True,24,Left Center Forward,93,1. Bundesliga,Germany,male,2023/2024,False,False
3,3895302,cf9088bc-7e59-4d57-8ac5-31658da858bb,1,6.728,176,31100.0,74.7250,36.295,74.7250,36.720,21,1,0,3,281,9,Regular Season,29,2024-04-14 17:30:00,904,176,5,0,BayArena,Harm Osmers,Werder Bremen,Christian Groß,,36,True,4,Center Back,93,1. Bundesliga,Germany,male,2023/2024,False,False
4,3895302,4464cb75-f45f-4508-8444-2560d1625d06,1,7.622,176,31100.0,74.7250,36.720,76.3875,53.805,0,1,5,4,281,9,Regular Season,29,2024-04-14 17:30:00,904,176,5,0,BayArena,Harm Osmers,Werder Bremen,Christian Groß,,36,True,4,Center Back,93,1. Bundesliga,Germany,male,2023/2024,False,False


In [16]:
df_labels.head()

,game_id,action_id,scores,concedes
0,3895302,0,False,False
1,3895302,1,False,False
2,3895302,2,False,False
3,3895302,3,False,False
4,3895302,4,False,False


In [17]:
import socceraction.spadl.config as spadlcfg

# Aggiungi bodypart_name
bodypart_map = {i: name for i, name in enumerate(spadlcfg.bodyparts)}
merged_df['bodypart_name'] = merged_df['bodypart_id'].map(bodypart_map)

# Aggiungi result_name
result_map = {i: name for i, name in enumerate(spadlcfg.results)}
merged_df['result_name'] = merged_df['result_id'].map(result_map)

# Aggiungi type_name
type_map = {i: name for i, name in enumerate(spadlcfg.actiontypes)}
merged_df['type_name'] = merged_df['type_id'].map(type_map)


print("Colonne dopo l'aggiunta dei nomi:", merged_df.columns.tolist())
merged_df[['action_id', 'bodypart_id', 'bodypart_name', 'result_id', 'result_name', 'type_id', 'type_name']].head()

Colonne dopo l'aggiunta dei nomi: ['game_id', 'original_event_id', 'period_id', 'time_seconds', 'team_id', 'player_id', 'start_x', 'start_y', 'end_x', 'end_y', 'type_id', 'result_id', 'bodypart_id', 'action_id', 'season_id', 'competition_id', 'competition_stage', 'game_day', 'game_date', 'home_team_id', 'away_team_id', 'home_score', 'away_score', 'venue', 'referee', 'team_name', 'player_name', 'nickname', 'jersey_number', 'is_starter', 'starting_position_id', 'starting_position_name', 'minutes_played', 'competition_name', 'country_name', 'competition_gender', 'season_name', 'scores', 'concedes', 'bodypart_name', 'result_name', 'type_name']


,action_id,bodypart_id,bodypart_name,result_id,result_name,type_id,type_name
0,0,5,foot_right,1,success,0,pass
1,1,0,foot,1,success,21,dribble
2,2,5,foot_right,1,success,0,pass
3,3,0,foot,1,success,21,dribble
4,4,5,foot_right,1,success,0,pass
